In [1]:
## Load libraries
import pandas as pd
import os
import rasterio
import numpy as np

import os
os.getcwd()

'/Users/henryoliver/Documents/MEDS/capstone/aridgw_cdl_exploration'

In [2]:
# Read in groundwater data
df = pd.read_csv("gw_data/clean_site_waterdata_04222026.csv")
df['latitude'] = df['latitude'].round(5)
df['longitude'] = df['longitude'].round(4)

In [3]:
# Set categories for land use

# Cultivated: codes 1-60 (minus 61 fallow), 66-77 (tree crops), 204-254 (specialty crops)
cultivated_codes = set(
    list(range(1, 61)) +
    list(range(66, 78)) +
    list(range(204, 255))
) - {61} # fallow

In [ ]:
## Results for all years 2008-2020


from pyproj import Transformer
from rasterio.windows import from_bounds
import pandas as pd
import numpy as np
import rasterio

all_results = []
for year in range(2008, 2021):
    print(f"Processing {year}...")
    df_year = df[df['year_value'] == year].copy()
    if len(df_year) == 0:
        continue
    tif_path = f"data/cultivation/{year}_30m_cdls/{year}_30m_cdls.tif"
    with rasterio.open(tif_path) as src:
        transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        for _, row in df_year.iterrows():
            x, y = transformer.transform(row['longitude'], row['latitude'])
            bbox = (x - 500, y - 500, x + 500, y + 500)
            window = from_bounds(*bbox, transform=src.transform)
            block = src.read(1, window=window)

            total = block.size
            cultivated = np.sum(np.isin(block, list(cultivated_codes)))
          #  pasture_rangeland = np.sum(np.isin(block, list(pasture_rangeland_codes)))
          #  water = np.sum(np.isin(block, list(water_codes)))
          #  no_data = np.sum(np.isin(block, list(no_data_codes)))
          #  other = total - cultivated - pasture_rangeland - water - no_data

            all_results.append({
                'latitude': round(row['latitude'], 5),
                'longitude': round(row['longitude'], 4),
                'year_value': year,
                'pct_cultivated': cultivated / total * 100,
               # 'pct_pasture_rangeland': pasture_rangeland / total * 100,
               # 'pct_water': water / total * 100,
               # 'pct_no_data': no_data / total * 100,
               # 'pct_other': other / total * 100,
            })

all_results_df = pd.DataFrame(all_results)

all_results_df = all_results_df.drop_duplicates(subset=['latitude', 'longitude', 'year_value'])


all_results_df.head()

Processing 2008...
Processing 2009...
Processing 2010...
Processing 2011...
Processing 2012...
Processing 2013...
Processing 2014...
Processing 2015...
Processing 2016...
Processing 2017...
Processing 2018...
Processing 2019...
Processing 2020...


,latitude,longitude,year_value,pct_cultivated
0,37.31502,-100.8505,2008,86.501377
1,37.34495,-101.6104,2008,32.966024
2,37.42949,-100.2434,2008,82.460973
3,37.56096,-98.0580,2008,77.043159
4,37.59874,-100.9497,2008,37.832874


In [5]:
df_final = df.merge(all_results_df, on=['latitude', 'longitude', 'year_value'], how='left')
gwl_cult_1km = df_final[['year_value', 'site_id', 'depth_to_gw_ft', 'depth_to_gw_m', 'latitude', 'longitude', 'data_source', 'region', 'pct_cultivated']]
gwl_cult_1km.tail()

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,region,pct_cultivated
986,2015,southern_willcox,132.500004,40.38600,31.36597,-109.663,Jasechko,SoCal_Arizona,0.000000
987,2016,southern_willcox,134.700004,41.05656,31.36597,-109.663,Jasechko,SoCal_Arizona,0.000000
988,2018,southern_willcox,147.000005,44.80560,31.36597,-109.663,Jasechko,SoCal_Arizona,14.325069
989,2019,southern_willcox,153.000005,46.63440,31.36597,-109.663,Jasechko,SoCal_Arizona,3.856749
990,2020,southern_willcox,154.700005,47.15256,31.36597,-109.663,Jasechko,SoCal_Arizona,3.305785


In [7]:
gwl_cult_1km.to_csv("outputs/gwl_cult_1km.csv", index=False)